In [1]:
from dataclasses import dataclass
import time

import numpy as np
import primme
from scipy.sparse.linalg import LinearOperator

import libdet
from pyscf import ao2mo, gto, scf

In [2]:
def davidson_primme(
    ham,
    dets: np.ndarray,
    guess: np.ndarray | None = None,
    *,
    tol: float = 1e-8,
    max_iter: int = 200,
    max_space: int = 64,
    mode: str = "sparse",
):
    t0 = time.perf_counter()

    dets = libdet.to_dets(dets)
    hdiag = np.asarray(ham.diags(dets), dtype=np.float64).reshape(-1)
    n = hdiag.size

    if n == 1:
        t_total = time.perf_counter() - t0
        return (
            float(hdiag[0]),
            np.array([1.0], dtype=np.float64),
            hdiag,
            0.0,
            0.0,
            t_total,
        )

    if guess is None:
        v0 = np.zeros(n, dtype=np.float64)
        v0[0] = 1.0
    else:
        v0 = np.asarray(guess, dtype=np.float64).reshape(-1).copy()
        v0 /= np.linalg.norm(v0)

    t_mat = time.perf_counter()

    if mode == "sparse":
        A = ham.matrix(dets, dets)
    elif mode == "matvec":
        def _matvec(x):
            x = np.asarray(x, dtype=np.float64).reshape(-1)
            return np.asarray(ham.matvec(dets, x, kets=dets), dtype=np.float64)

        def _matmat(X):
            X = np.asarray(X, dtype=np.float64)
            return np.asarray(ham.matvec(dets, X, kets=dets), dtype=np.float64)

        A = LinearOperator(
            shape=(n, n),
            matvec=_matvec,
            matmat=_matmat,
            dtype=np.float64,
        )
    else:
        raise ValueError("mode must be 'sparse' or 'matvec'")

    t_mat = time.perf_counter() - t_mat

    def _precond(x):
        X = np.asarray(x, dtype=np.float64)
        is_vec = X.ndim == 1
        if is_vec:
            X = X[:, None]

        shifts = np.asarray(
            primme.get_eigsh_param("ShiftsForPreconditioner"),
            dtype=np.float64,
        )
        if shifts.size == 0:
            shifts = np.zeros(X.shape[1], dtype=np.float64)
        elif shifts.size == 1 and X.shape[1] > 1:
            shifts = np.full(X.shape[1], shifts[0], dtype=np.float64)

        Y = np.empty_like(X)
        for j in range(X.shape[1]):
            denom = hdiag - shifts[j]
            denom = np.where(
                np.abs(denom) < 1e-8,
                np.copysign(1e-8, denom),
                denom,
            )
            Y[:, j] = X[:, j] / denom

        return Y[:, 0] if is_vec else Y

    OPinv = LinearOperator(
        shape=(n, n),
        matvec=_precond,
        matmat=_precond,
        dtype=np.float64,
    )

    ncv = min(n, max(8, min(max_space, 24)))

    t_solve = time.perf_counter()
    w, v = primme.eigsh(
        A,
        k=1,
        which="SA",
        v0=v0[:, None],
        OPinv=OPinv,
        tol=tol,
        maxiter=max_iter,
        ncv=ncv,
        maxBlockSize=1,
        raise_for_unconverged=False,
    )
    t_solve = time.perf_counter() - t_solve

    w = np.asarray(w, dtype=np.float64).reshape(-1)
    v = np.asarray(v, dtype=np.float64)

    if w.size == 0:
        c = v0.copy()
        Ac = A @ c
        e = float(np.dot(c, Ac))
        t_other = time.perf_counter() - t0 - t_mat - t_solve
        return e, c, hdiag, t_mat, t_solve, t_other

    c = v[:, 0].copy()
    if c[np.argmax(np.abs(c))] < 0.0:
        c = -c
    c /= np.linalg.norm(c)

    t_other = time.perf_counter() - t0 - t_mat - t_solve
    return float(w[0]), c, hdiag, t_mat, t_solve, t_other

In [3]:
@dataclass(slots=True)
class State:
    dets: np.ndarray
    coeffs: np.ndarray
    energy: float
    diags: np.ndarray
    eps: float | None = None


def hf_det(norb: int, nelec: tuple[int, int]) -> np.ndarray:
    nword = (norb + 63) // 64
    det = np.zeros((2, nword), dtype=np.uint64)

    for spin, nocc in enumerate(nelec):
        for p in range(nocc):
            det[spin, p // 64] |= np.uint64(1) << np.uint64(p % 64)

    return det


def hci_solve(
    ham,
    nelec: tuple[int, int],
    *,
    eps: float = 1e-4,
    max_cycle: int = 10,
) -> State:
    total_start = time.perf_counter()

    det0 = hf_det(int(ham.norb), nelec)
    dets = det0[np.newaxis]
    coeffs = np.array([1.0], dtype=np.float64)
    diags = ham.diags(dets)
    energy = float(diags[0])

    header = (
        f"{'Iter':>4} | {'Ndet':>8} | {'Screen (s)':>12} | "
        f"{'Matrix (s)':>12} | {'Solve (s)':>11} | {'OI (s)':>8} | {'Energy':>16}"
    )
    print(header)
    print("-" * len(header))

    for i in range(max_cycle):
        t_screen = time.perf_counter()

        p_dets = ham.expand(
            dets,
            eps,
            coeffs=coeffs,
            exclude=dets,
        )
        t_screen = time.perf_counter() - t_screen

        n_old = len(dets)
        if p_dets.shape[0] == 0:
            print(f"Converged at cycle {i}: no new determinants.")
            break

        t_dets = np.concatenate([dets, p_dets], axis=0)

        guess = np.zeros(t_dets.shape[0], dtype=np.float64)
        guess[:n_old] = coeffs

        dets = np.ascontiguousarray(t_dets, dtype=np.uint64)
        energy, coeffs, diags, t_mat, t_solve, t_other = davidson_primme(ham, dets, guess)

        print(
            f"{i + 1:4d} | {len(dets):8d} | {t_screen:12.4f} | "
            f"{t_mat:12.4f} | {t_solve:11.4f} | {t_other:8.4f} | {energy:16.10f}"
        )

    print("-" * len(header))
    print(f"Total: {time.perf_counter() - total_start:.4f} s")

    return State(dets=dets, coeffs=coeffs, energy=energy, diags=diags, eps=float(eps))

In [4]:
"""
t_graph: time to build sparse H
t_solve: time for PRIMME Davidson
t_other: remaining overhead
"""

mol = gto.M(
    atom=
    '''
    O   0.00000000,  0.00000000,  0.00000000
    H   0.75700000,  0.00000000,  0.58590000
    H  -0.75700000,  0.00000000,  0.58590000
    ''',
    # atom=
    # '''
    # N   0.00000000,  0.00000000,  0.00000000
    # N   0.00000000,  0.00000000,  4.20000000
    # ''',
    basis="cc-pvdz",
    unit="Angstrom",
    verbose=0,
)
mf = scf.RHF(mol).run()

norb = mf.mo_coeff.shape[1]
nelec = mol.nelec
h1e = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
eri = ao2mo.restore(8, ao2mo.kernel(mol, mf.mo_coeff), norb)

ham   = libdet.Hamiltonian.rhf(h1e, eri, ecore=mol.energy_nuc())
state = hci_solve(ham, nelec, eps=5e-4)

print(f"SCF      : {mf.e_tot:.12f}")
print(f"HCI var  : {state.energy:.12f}  Ndet: {len(state.dets)}")

Iter |     Ndet |   Screen (s) |   Matrix (s) |   Solve (s) |   OI (s) |           Energy
-----------------------------------------------------------------------------------------
   1 |     3041 |       0.0237 |       0.0644 |      0.0301 |   0.0086 |   -76.2312761110
   2 |    20738 |       0.0272 |       0.4539 |      0.0823 |   0.0255 |   -76.2391388079
   3 |    22823 |       0.0963 |       0.4930 |      0.0923 |   0.0337 |   -76.2394213178
   4 |    22917 |       0.1118 |       0.4805 |      0.0697 |   0.0293 |   -76.2394307247
   5 |    22921 |       0.1014 |       0.4841 |      0.0656 |   0.0262 |   -76.2394310106
Converged at cycle 5: no new determinants.
-----------------------------------------------------------------------------------------
Total: 2.9145 s
SCF      : -76.026796341400
HCI var  : -76.239431010629  Ndet: 22921


In [5]:
def semi_pt2(
    ham,
    state: State,
    *,
    eps1: float = 1e-4,
    eps2: float = 1e-12,
    counts: int = 4,
    n_rep: int = 16,
    seed: int = 0,
) -> tuple[float, float, float, float]:

    dets = state.dets
    coeffs = state.coeffs
    energy = state.energy

    # Deterministic strong-space PT2 at eps1.
    p_dets = ham.expand(dets, eps1, coeffs=coeffs, exclude=dets)
    prj = ham.project(p_dets, dets, coeffs, eps=eps1)
    hpsi = np.asarray(prj.hpsi, dtype=np.float64)
    diags = np.asarray(prj.diags, dtype=np.float64)

    e2_det = float(np.sum((hpsi * hpsi) / (energy - diags)))

    # Weak-shell correction in eps2 <= |H_ai c_i| < eps1.
    shell = ham.sample_shell(
        dets,
        coeffs,
        eps1,
        eps2,
        counts,
        exclude=dets,
        n_rep=n_rep,
        seed=seed,
    )

    rep_ptr = np.asarray(shell.rep_ptr, dtype=np.int64)
    diags = np.asarray(shell.diags, dtype=np.float64)
    hpsi_strong = np.asarray(shell.hpsi_strong, dtype=np.float64)
    hpsi_a = np.asarray(shell.hpsi_a, dtype=np.float64)
    hpsi_b = np.asarray(shell.hpsi_b, dtype=np.float64)

    corr = np.zeros(n_rep, dtype=np.float64)
    for r in range(n_rep):
        lo = int(rep_ptr[r])
        hi = int(rep_ptr[r + 1])
        if hi == lo:
            continue

        denom = energy - diags[lo:hi]
        s = hpsi_strong[lo:hi]
        wa = hpsi_a[lo:hi]
        wb = hpsi_b[lo:hi]
        corr[r] = np.sum((s * (wa + wb) + wa * wb) / denom)

    e2_stoch = float(np.mean(corr))
    err = 0.0 if n_rep == 1 else float(np.std(corr, ddof=1) / np.sqrt(n_rep))
    return e2_det + e2_stoch, e2_det, e2_stoch, err

e2_total, e2_det, e2_stoch, err = semi_pt2(ham, state)
total_energy = state.energy + e2_total

print(f"Variational: {state.energy:16.12f}")
print(f"PT2: {e2_total:14.12f} +/- {err:14.12f}")
print(f"Total: {total_energy:16.12f} +/- {err:14.12f}")

Variational: -76.239431010629
PT2: -0.004451766954 +/- 0.000088181627
Total: -76.243882777583 +/- 0.000088181627
